# Module 7 (streaming) - create the Eventstream and stream live transactions

Everything here runs **from the notebook**:
1. Creates the Eventstream `NorthwindTxnStream` (Custom endpoint source to Eventhouse `LiveTransactions`) via the Fabric REST API.
2. Reads the Custom endpoint's Event Hub-compatible **connection string** privately from the API.
3. Streams enriched transaction observations with fresh UTC timestamps and existing transaction IDs using the `azure-eventhub` SDK.
4. Confirms the send-window count and exact reference event in `LiveTransactions`.

**Prereqs:** the Eventhouse `NorthwindRTI` and table `LiveTransactions` must already exist. Do not rerun the batch setup notebook when the table already contains data.
The **NorthwindLakehouse** lakehouse must be the default lakehouse (source CSVs live in `Files/NorthwindLakehouse/payments`).
Install packages before configuration because `%pip` can restart the Python interpreter.

In [ ]:
%pip install azure-eventhub --quiet

In [ ]:
ES_NAME = "NorthwindTxnStream"
SRC_NAME = "NorthwindTxnSource"
EH_DB = "NorthwindRTI"
TABLE = "LiveTransactions"
N_EVENTS = 300
BASE = "/lakehouse/default/Files/NorthwindLakehouse/payments"

WS = notebookutils.runtime.context["currentWorkspaceId"]
print("workspace:", WS)

In [ ]:
import json
import requests

FAB = "https://api.fabric.microsoft.com/v1"
def _h():
    return {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}", "Content-Type": "application/json", "x-ms-fabric-skill": "spark-cli"}

kdb = [d for d in requests.get(f"{FAB}/workspaces/{WS}/kqlDatabases", headers=_h()).json()["value"]
       if d["displayName"] == EH_DB][0]
KQLDB_ID = kdb["id"]
QUERY_URI = kdb["properties"]["queryServiceUri"]
print("KQL database:", KQLDB_ID)

In [ ]:
import base64, time

# idempotent: delete any prior Eventstream of this name
for e in requests.get(f"{FAB}/workspaces/{WS}/eventstreams", headers=_h()).json()["value"]:
    if e["displayName"] == ES_NAME:
        requests.delete(f"{FAB}/workspaces/{WS}/eventstreams/{e['id']}", headers=_h()); time.sleep(3)

topology = {
    "sources": [{"name": SRC_NAME, "type": "CustomEndpoint", "properties": {}}],
    "destinations": [{
        "name": "ToEventhouse", "type": "Eventhouse",
        "properties": {"dataIngestionMode": "ProcessedIngestion", "workspaceId": WS,
                       "itemId": KQLDB_ID, "databaseName": EH_DB, "tableName": TABLE,
                       "inputSerialization": {"type": "Json", "properties": {"encoding": "UTF8"}}},
        "inputNodes": [{"name": f"{ES_NAME}-stream"}]}],
    "operators": [],
    "streams": [{"name": f"{ES_NAME}-stream", "type": "DefaultStream", "properties": {},
                 "inputNodes": [{"name": SRC_NAME}]}],
    "compatibilityLevel": "1.0",
}
platform = {"$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
            "metadata": {"type": "Eventstream", "displayName": ES_NAME},
            "config": {"version": "2.0", "logicalId": "00000000-0000-0000-0000-000000000000"}}
def b64(o): return base64.b64encode(json.dumps(o).encode()).decode()

body = {"displayName": ES_NAME, "type": "Eventstream", "definition": {"parts": [
    {"path": "eventstream.json", "payload": b64(topology), "payloadType": "InlineBase64"},
    {"path": ".platform",        "payload": b64(platform), "payloadType": "InlineBase64"}]}}
r = requests.post(f"{FAB}/workspaces/{WS}/items", headers=_h(), json=body)
if r.status_code == 202:
    op = r.headers["x-ms-operation-id"]
    while True:
        time.sleep(5)
        st = requests.get(f"{FAB}/operations/{op}", headers=_h()).json()
        if st["status"] not in ("Running", "NotStarted"): break
    print("create:", st["status"])
    assert st["status"] == "Succeeded", st
else:
    r.raise_for_status()
time.sleep(4)
ES_ID = [e for e in requests.get(f"{FAB}/workspaces/{WS}/eventstreams", headers=_h()).json()["value"]
         if e["displayName"] == ES_NAME][0]["id"]
print("EVENTSTREAM_ID:", ES_ID)

In [ ]:
# wait until the Eventhouse destination is Running, then read the custom endpoint connection string
while True:
    topo = requests.get(f"{FAB}/workspaces/{WS}/eventstreams/{ES_ID}/topology", headers=_h()).json()
    if all(n["status"] == "Running" for n in topo["sources"] + topo["destinations"]): break
    time.sleep(5)
SRC_ID = [s for s in topo["sources"] if s["name"] == SRC_NAME][0]["id"]
conn = requests.get(f"{FAB}/workspaces/{WS}/eventstreams/{ES_ID}/sources/{SRC_ID}/connection", headers=_h()).json()
CONN_STR = conn["accessKeys"]["primaryConnectionString"]
print("custom endpoint ready:", conn["eventHubName"])

In [ ]:
import csv
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

def load(p):
    with open(f"{BASE}/{p}", encoding="utf-8") as f:
        return list(csv.DictReader(f))

home  = {c["customerId"]: c["homeCountry"] for c in load("customers.csv")}
mrisk = {m["merchantId"]: (m["riskFlag"] == "1") for m in load("merchants.csv")}
txns  = load("transactions.csv")
fraud = [t for t in txns if t["isFraud"] == "1"]
sample = (fraud + [t for t in txns if t["isFraud"] != "1"])[:N_EVENTS]

def event(t):
    return {"txnId": t["txnId"], "cardId": t["cardId"], "accountId": t["accountId"],
            "customerId": t["customerId"], "merchantId": t["merchantId"],
            "amount": float(t["amount"]), "currency": t["currency"],
            "eventTime": datetime.now(timezone.utc).isoformat(),
            "channel": t["channel"], "geoCountry": t["geoCountry"],
            "mcc": int(t["mcc"]) if t["mcc"] else None,
            "homeCountry": home.get(t["customerId"], ""),
            "merchantRisk": bool(mrisk.get(t["merchantId"], False)),
            "isFraud": t["isFraud"] == "1", "fraudType": t["fraudType"]}

SEND_STARTED = datetime.now(timezone.utc).isoformat()
FIRST_EVENT = None
producer = EventHubProducerClient.from_connection_string(CONN_STR)
sent = 0
with producer:
    batch = producer.create_batch()
    for t in sample:
        payload = event(t)
        if FIRST_EVENT is None:
            FIRST_EVENT = payload
        data = EventData(json.dumps(payload))
        try:
            batch.add(data)
        except ValueError:
            producer.send_batch(batch); sent += len(batch)
            batch = producer.create_batch(); batch.add(data)
    if len(batch):
        producer.send_batch(batch); sent += len(batch)
SEND_FINISHED = datetime.now(timezone.utc).isoformat()
print(f"streamed {sent} events into {ES_NAME}")
print("send window UTC:", SEND_STARTED, SEND_FINISHED)
if FIRST_EVENT:
    print("verification event:", FIRST_EVENT["txnId"], FIRST_EVENT["eventTime"], FIRST_EVENT["amount"])

## Confirm events arrived through the stream
ProcessedIngestion buffers, so give it up to ~2 minutes. Re-run the next cell until the send-window count and exact verification event appear. Do not rerun the producer just to check arrival.

In [ ]:
def kql(query):
    tok = notebookutils.credentials.getToken(QUERY_URI)
    r = requests.post(f"{QUERY_URI}/v1/rest/query",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"db": EH_DB, "csl": query}, timeout=60)
    r.raise_for_status()
    return r.json()["Tables"][0]["Rows"]

print("LiveTransactions total / latest ingestion:")
print(kql(f"{TABLE} | summarize total=count(), latest=max(ingestion_time())"))
print("Events in this send window:")
print(kql(f"{TABLE} | where eventTime between (datetime({SEND_STARTED}) .. datetime({SEND_FINISHED})) | summarize events=count(), transactions=dcount(txnId), latest=max(eventTime)"))
if FIRST_EVENT:
    print("Exact verification event:")
    print(kql(f"{TABLE} | where txnId == {json.dumps(FIRST_EVENT['txnId'])} and eventTime == datetime({FIRST_EVENT['eventTime']}) | project txnId, eventTime, amount, merchantId, isFraud, ingested=ingestion_time()"))